# Encoder Pipeline — Full SimCLR Pretraining  (HPC training twin)

Self-supervised **SimCLR** pretraining of the shared **ConvNeXtV2** bi-planar encoder on the Sunway
HPC GPU. This is the twin of `notebooks/modeling/encoder_pipeline.ipynb`; the **only** differences
are the CONFIG cell (CUDA device, full epochs, mixed precision, data workers) and a real training
loop in place of the local 10-step smoke.

It produces `models/convnextv2_simclr_encoder.pth` — the encoder checkpoint that **both**
`decoder_pipeline.ipynb` twins load via `fusion.load_simclr_encoder(...)`. For the k-fold comparison
SimCLR pretrains **once on ALL knees** (it uses no labels) and the single fold-agnostic encoder is
reused across folds; the label-using stages (per-fold front-end and decoder) are what stay within
their fold, so the comparison has no **label** leakage (see the §2 cross-validation note).

Pipeline: `AP+LAT DRRs -> two augmented views -> ConvNeXtV2 -> projector -> NT-Xent contrastive loss`

### How to run (HPC)

```bash
cd "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
source .venv/bin/activate
pip install -r requirements.txt
```

Then run the cells top to bottom. The **CONFIG** cell is the only place you change settings. Set
`SMOKE_TEST = True` first to dry-run the whole notebook quickly (CPU-friendly) before the full GPU
run.

In [ ]:
import os, math, random, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# ============================= CONFIG (HPC - full SimCLR pretraining) =============================
# Mirrors the local encoder notebook; only these knobs differ. Run on the Sunway HPC GPU.
ENV          = "HPC"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE     = 256
EPOCHS       = 10
BATCH_SIZE   = 128           # SimCLR likes many negatives; lower if the GPU runs out of memory
LR           = 3e-4
WARMUP_EPOCHS = 10
TEMPERATURE  = 0.5
PROJ_DIM     = 128
NUM_WORKERS  = 4
USE_AMP      = True
CKPT_EVERY   = 25
PRETRAINED   = True
FREEZE_ENCODER = False        # SimCLR always trains the full encoder; kept for signature parity
SPLIT_FRACS  = (0.70, 0.15, 0.15)   # knee-level split, identical to decoder_pipeline.ipynb
INCLUDE_GEOMETRIC = True       # SSL has no 3D GT, so geometric variants are safe extra views
SMOKE_TEST   = False
SMOKE_EPOCHS = 2
SMOKE_STEPS  = 10
SMOKE_CASES_PER_GROUP = 3
RESUME_FROM  = None
EXPLICIT_ROOT = None          # e.g. "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
if DEVICE.type != "cuda":
    print("[warning] CUDA not available - this HPC notebook expects a GPU (CPU is fine only for SMOKE_TEST).")
print("ENV", ENV, "| device", DEVICE, "| epochs", EPOCHS, "| batch", BATCH_SIZE, "| smoke", SMOKE_TEST)

In [ ]:
# Resolve the project root robustly (works locally and on HPC, regardless of where the notebook is
# launched from). We look upward for data/interim/predrr (the GT CT folder), matching
# decoder_pipeline.ipynb so both notebooks resolve the same ROOT.
def find_root(start: Path) -> Path:
    if EXPLICIT_ROOT:
        r = Path(EXPLICIT_ROOT)
        if (r / "data" / "interim" / "predrr").exists():
            return r
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "data" / "interim" / "predrr").exists():
            return cand
    raise FileNotFoundError("Could not find project root (expected data/interim/predrr). "
                            "Set EXPLICIT_ROOT in the CONFIG cell.")

ROOT            = find_root(Path.cwd())
DATA            = ROOT / "data"
NORMAL_DRR_DIR  = DATA / "interim" / "DRRs"                 # normal AP/LAT DRRs
AUG_DRR_DIR     = DATA / "processed" / "augmented_DRRs"     # augmented DRR variants
PREDRR_DIR      = DATA / "interim" / "predrr"               # GT CT (only anchors ROOT here)
MODELS_DIR      = ROOT / "models"; MODELS_DIR.mkdir(exist_ok=True)
SIMCLR_CKPT        = MODELS_DIR / "convnextv2_simclr_encoder.pth"      # <- the contract file the decoder loads
BEST_CKPT          = MODELS_DIR / "convnextv2_simclr_encoder_best.pth"
TRAINSTATE_CKPT    = MODELS_DIR / "encoder_simclr_trainstate.pth"      # full state for RESUME_FROM
HISTORY_CSV        = MODELS_DIR / "encoder_simclr_history.csv"
PRETRAIN_SPLIT_CSV = MODELS_DIR / "simclr_pretrain_split.csv"
print("ROOT:", ROOT)
print("encoder checkpoint ->", SIMCLR_CKPT)

## 1. Shared encoder + SimCLR head

The encoder front-end below is **copied verbatim** from `encoder_pipeline.ipynb` / the decoder
twin, so the `encoder.state_dict()` this notebook saves loads back into `BiPlanarFeatureFusion`
with `missing=0 unexpected=0`. **Do not edit it here.**

On top of it we add the SimCLR pieces: `SimCLRModel` (encoder -> global-average-pool the deepest
map -> projector -> L2-normalised embedding) and the corrected `nt_xent_loss` (the reference's
labels pointed each anchor at its own masked diagonal; here row `i` is positive with row `i+B`).

In [ ]:
# ===== Encoder front-end - VERBATIM from encoder_pipeline.ipynb. DO NOT EDIT. =====
BACKBONE     = "convnextv2_tiny"
OUT_CHANNELS = [64, 128, 256, 512]
FUSION_TYPES = ["local", "local", "attn", "attn"]   # fine -> coarse

def make_backbone(pretrained=True):
    """features_only ConvNeXtV2 returning 4 multi-scale maps. Falls back to random init offline."""
    try:
        return timm.create_model(BACKBONE, pretrained=pretrained, features_only=True)
    except Exception as e:
        print("[warn] pretrained fetch failed (%s); random init." % type(e).__name__)
        return timm.create_model(BACKBONE, pretrained=False, features_only=True)

FEAT_DIMS = [f["num_chs"] for f in make_backbone(pretrained=False).feature_info]   # [96,192,384,768]

def load_drr(path):
    """npy 256x256 float32 [0,1] -> tensor [3,H,W] (1 channel replicated to 3 for ConvNeXtV2)."""
    arr = np.load(path).astype(np.float32)
    t = torch.from_numpy(arr)
    if t.ndim == 2:
        t = t.unsqueeze(0)
    return t.repeat(3, 1, 1) if t.shape[0] == 1 else t

NORMALIZE = T.Normalize(mean=[0.5] * 3, std=[0.5] * 3)
def paired_tf(t):
    return NORMALIZE(t)

class CrossAttention(nn.Module):
    """AP (query) attends to LAT (key/value). Operates on tokens [B, N, C]."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim); self.k = nn.Linear(dim, dim); self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5
    def forward(self, a, b):
        attn = F.softmax(torch.matmul(self.q(a), self.k(b).transpose(-2, -1)) * self.scale, dim=-1)
        return torch.matmul(attn, self.v(b)) + a

class LocalFusion(nn.Module):
    """Cheap high-res fusion: concat views + 3x3 conv, residual on AP."""
    def __init__(self, dim):
        super().__init__()
        self.mix = nn.Conv2d(2 * dim, dim, kernel_size=3, padding=1)
    def forward(self, a, b):
        return self.mix(torch.cat([a, b], dim=1)) + a

class BiPlanarFeatureFusion(nn.Module):
    def __init__(self, feat_dims=FEAT_DIMS, out_channels=OUT_CHANNELS,
                 fusion_types=FUSION_TYPES, depth=16, pretrained=True, freeze_encoder=False):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        self.fusion_types = list(fusion_types); self.depth = depth
        self.fuse = nn.ModuleList([CrossAttention(d) if t == "attn" else LocalFusion(d)
                                   for d, t in zip(feat_dims, fusion_types)])
        self.to3d = nn.ModuleList([nn.Conv2d(c, o, 1) for c, o in zip(feat_dims, out_channels)])
        self.expand3d = nn.ModuleList([nn.Conv3d(o, o, 3, padding=1) for o in out_channels])
        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
    def load_simclr_encoder(self, path):
        missing, unexpected = self.encoder.load_state_dict(torch.load(path, map_location="cpu"), strict=False)
        print("loaded SimCLR encoder: missing=%d unexpected=%d" % (len(missing), len(unexpected)))
    def forward(self, ap_img, lat_img):
        ap_feats, lat_feats = self.encoder(ap_img), self.encoder(lat_img)
        fused2d, fused3d = [], []
        for ap_f, lat_f, fuse, c2d, c3d, t in zip(
                ap_feats, lat_feats, self.fuse, self.to3d, self.expand3d, self.fusion_types):
            B, C, H, W = ap_f.shape
            if t == "attn":
                a = ap_f.flatten(2).transpose(1, 2); b = lat_f.flatten(2).transpose(1, 2)
                f2d = fuse(a, b).transpose(1, 2).reshape(B, C, H, W)
            else:
                f2d = fuse(ap_f, lat_f)
            fused2d.append(f2d)
            f3 = c2d(f2d).unsqueeze(2)
            f3 = F.interpolate(f3, size=(self.depth, H, W), mode="trilinear", align_corners=False)
            fused3d.append(c3d(f3))
        return fused2d, fused3d

# ===== SimCLR head (corrected NT-Xent; validated in encoder_pipeline.ipynb) =====
class SimCLRModel(nn.Module):
    def __init__(self, proj_dim=128, pretrained=True):
        super().__init__()
        self.encoder = make_backbone(pretrained)
        fd = self.encoder.feature_info[-1]["num_chs"]
        self.projector = nn.Sequential(nn.Linear(fd, fd), nn.ReLU(), nn.Linear(fd, proj_dim))
    def forward(self, x):
        pooled = self.encoder(x)[-1].mean(dim=(2, 3))   # GAP on the deepest map -> [B, fd]
        return F.normalize(self.projector(pooled), dim=1)

def nt_xent_loss(z1, z2, temperature=0.5):
    B = z1.size(0)
    z = torch.cat([z1, z2], dim=0)
    sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2) / temperature
    labels = torch.cat([torch.arange(B) + B, torch.arange(B)]).to(z.device)   # i <-> i+B
    sim.masked_fill_(torch.eye(2 * B, dtype=torch.bool, device=z.device), -9e15)
    return F.cross_entropy(sim, labels)

print("encoder feature dims:", FEAT_DIMS)

## 2. Data — all-knee DRRs for self-supervised pretraining

We index both normal and augmented DRRs. Each AP and each LAT image becomes a SimCLR anchor;
`SimCLRDRRDataset` returns two stochastic views of it.

> **Cross-validation note.** SimCLR uses **no occupancy labels**, so for the k-fold comparison we
> pretrain it **once on ALL knees** and reuse the single fold-agnostic encoder. This eliminates
> **label** leakage in the comparison (the per-fold front-end and decoder are the only label-using
> stages, and they stay within their fold); the residual is a mild *representation* leakage (the
> encoder has seen each fold's test *images*, never its labels) — the standard accepted trade-off in
> low-data regimes. The list of pretraining knees is written to `simclr_pretrain_split.csv` for audit.

In [ ]:
def build_paired_index():
    """One row per (case, side, variant) with absolute AP/LAT paths + metadata."""
    rows = []
    nmeta = pd.read_csv(NORMAL_DRR_DIR / "drr_generation_metadata.csv")
    for (ds, case, side), _ in nmeta.groupby(["dataset", "case", "side"]):
        ap = NORMAL_DRR_DIR / ds / case / side / "ap.npy"
        lat = NORMAL_DRR_DIR / ds / case / side / "lat.npy"
        if ap.exists() and lat.exists():
            rows.append(dict(dataset=ds, case=case, side=side, variant="normal",
                             geometric=False, ap=str(ap), lat=str(lat)))
    ameta_path = AUG_DRR_DIR / "augmentation_variants_metadata.csv"
    if ameta_path.exists():
        ameta = pd.read_csv(ameta_path)
        for r in ameta.itertuples(index=False):
            ap = AUG_DRR_DIR / r.ap_npy; lat = AUG_DRR_DIR / r.lat_npy
            if ap.exists() and lat.exists():
                rows.append(dict(dataset=r.dataset, case=r.case, side=r.side, variant=r.variant,
                                 geometric=bool(r.geometric), ap=str(ap), lat=str(lat)))
    return pd.DataFrame(rows)

class SimCLRDRRDataset(Dataset):
    """Every AP and LAT image is an anchor; returns two stochastic views + its dataset label."""
    def __init__(self, df, aug):
        recs = []
        for r in df.itertuples(index=False):
            recs.append((r.ap, r.dataset)); recs.append((r.lat, r.dataset))
        self.recs = recs; self.aug = aug
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        path, ds = self.recs[i]
        base = load_drr(path)
        return self.aug(base), self.aug(base), ds

paired_index = build_paired_index()
if not INCLUDE_GEOMETRIC:
    paired_index = paired_index[~paired_index.geometric].reset_index(drop=True)
# CROSS-VALIDATION: SimCLR is self-supervised (no occupancy labels), so for the k-fold comparison we
# pretrain it ONCE on ALL knees and reuse the single fold-agnostic encoder across folds. This keeps
# the label-using stages (per-fold front-end + decoder) strictly within their fold (no LABEL leakage);
# the encoder seeing every knee's images is a mild, documented representation-leakage trade-off. To
# build a per-fold encoder instead, restrict train_df to that fold's train knees and fold-tag SIMCLR_CKPT.
train_df = paired_index.reset_index(drop=True)
if SMOKE_TEST:
    train_df = train_df.groupby("dataset").head(SMOKE_CASES_PER_GROUP).reset_index(drop=True)
train_df[["dataset", "case", "side", "variant"]].to_csv(PRETRAIN_SPLIT_CSV, index=False)
print("paired rows:", len(paired_index), "| SSL rows (all knees):", len(train_df))
print("cases:", train_df.groupby("dataset")["case"].nunique().to_dict(),
      "| anchor images:", 2 * len(train_df))

## 3. Two-view augmentation (bone X-ray appropriate)

Grayscale, so **no colour jitter on hue/saturation**. We use geometric (resized crop, flip, small
rotation) and intensity (blur, brightness/contrast) perturbations — the kinds of variation a knee
radiograph realistically shows. Two independent draws of this pipeline give the positive pair
SimCLR contrasts against the rest of the batch.

In [ ]:
simclr_aug = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), antialias=True),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    NORMALIZE,
])
print("two-view SimCLR augmentation ready")

## 4. SimCLR pretraining (full run with checkpointing)

`AdamW` with a linear **warmup** then **cosine** decay; mixed precision on GPU (bf16 where
available, else fp16 + GradScaler); gradient clipping at 1.0. Each epoch we log the mean NT-Xent
loss to `encoder_simclr_history.csv` and write checkpoints:
- `convnextv2_simclr_encoder.pth` — the **encoder weights** the decoder loads (overwritten each
  epoch as the latest; replaced by the best at the end),
- `convnextv2_simclr_encoder_best.pth` — best epoch (lowest loss),
- `convnextv2_simclr_encoder_epochNNN.pth` — every `CKPT_EVERY` epochs,
- `encoder_simclr_trainstate.pth` — full model/optimizer/scheduler state for `RESUME_FROM`.

Set `SMOKE_TEST = True` to run a couple of short epochs (CPU-friendly) end-to-end first.

In [ ]:
bs = 4 if SMOKE_TEST else BATCH_SIZE
nw = 0 if SMOKE_TEST else NUM_WORKERS
simclr_ds = SimCLRDRRDataset(train_df, aug=simclr_aug)
simclr_dl = DataLoader(simclr_ds, batch_size=bs, shuffle=True, num_workers=nw,
                       drop_last=True, pin_memory=(DEVICE.type == "cuda"))
print("anchor images:", len(simclr_ds), "| batch:", bs, "| steps/epoch:", len(simclr_dl))

simclr = SimCLRModel(proj_dim=PROJ_DIM, pretrained=PRETRAINED).to(DEVICE)
optimizer = torch.optim.AdamW(simclr.parameters(), lr=LR)

EPOCHS_EFF = SMOKE_EPOCHS if SMOKE_TEST else EPOCHS
def lr_factor(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / max(1, WARMUP_EPOCHS)
    prog = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS_EFF - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

use_amp = USE_AMP and DEVICE.type == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = use_amp and amp_dtype == torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

start_epoch, best_loss, history = 0, float("inf"), []
if RESUME_FROM:
    ck = torch.load(RESUME_FROM, map_location=DEVICE)
    simclr.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    if ck.get("scheduler"):
        scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1; best_loss = ck.get("best_loss", float("inf"))
    print("resumed from %s at epoch %d" % (RESUME_FROM, start_epoch))

t0 = time.time()
for epoch in range(start_epoch, EPOCHS_EFF):
    simclr.train(); running, nb = 0.0, 0
    for step, (v1, v2, _) in enumerate(simclr_dl):
        if SMOKE_TEST and step >= SMOKE_STEPS:
            break
        v1 = v1.to(DEVICE, non_blocking=True); v2 = v2.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.amp.autocast("cuda", dtype=amp_dtype):
                loss = nt_xent_loss(simclr(v1), simclr(v2), TEMPERATURE)
        else:
            loss = nt_xent_loss(simclr(v1), simclr(v2), TEMPERATURE)
        if use_scaler:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(simclr.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(simclr.parameters(), 1.0)
            optimizer.step()
        running += loss.item(); nb += 1
    scheduler.step()
    avg = running / max(1, nb); lr_now = optimizer.param_groups[0]["lr"]
    history.append(dict(epoch=epoch, loss=avg, lr=lr_now, secs=round(time.time() - t0, 1)))
    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)
    torch.save(simclr.encoder.state_dict(), SIMCLR_CKPT)   # latest encoder (contract file)
    torch.save({"epoch": epoch, "model": simclr.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "best_loss": best_loss,
                "config": {"BACKBONE": BACKBONE, "PROJ_DIM": PROJ_DIM, "TEMPERATURE": TEMPERATURE,
                           "IMG_SIZE": IMG_SIZE}}, TRAINSTATE_CKPT)
    if avg < best_loss:
        best_loss = avg
        torch.save(simclr.encoder.state_dict(), BEST_CKPT)
    if epoch % CKPT_EVERY == 0:
        torch.save(simclr.encoder.state_dict(),
                   MODELS_DIR / ("convnextv2_simclr_encoder_epoch%03d.pth" % epoch))
    print("epoch %03d/%d | nt_xent %.4f | lr %.2e | best %.4f"
          % (epoch, EPOCHS_EFF, avg, lr_now, best_loss))

if BEST_CKPT.exists():
    shutil.copyfile(BEST_CKPT, SIMCLR_CKPT)   # the decoder should load the best encoder
print("done. encoder checkpoint ->", SIMCLR_CKPT, "| best loss %.4f" % best_loss)

## 5. Checkpoint contract validation

Load the saved encoder into `BiPlanarFeatureFusion` (the class the decoder uses) and confirm a
clean transfer (`missing=0 unexpected=0`), then forward one healthy and one fractured real DRR pair
and assert the four 3D feature shapes equal the decoder contract.

In [ ]:
fusion = BiPlanarFeatureFusion(depth=16, pretrained=PRETRAINED, freeze_encoder=True).to(DEVICE).eval()
if SIMCLR_CKPT.exists():
    fusion.load_simclr_encoder(SIMCLR_CKPT)

def first_pair(df, dataset):
    sub = df[df.dataset == dataset]
    return sub.iloc[0] if len(sub) else None

expected = [(c, IMG_SIZE // s) for c, s in zip(OUT_CHANNELS, [4, 8, 16, 32])]
samples = {ds: first_pair(paired_index, ds) for ds in ["healthy", "fractured"]}
for ds, r in samples.items():
    if r is None:
        print("[skip] no %s pair" % ds); continue
    ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
    lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, f3d = fusion(ap, lat)
    print("\n%s: %s %s (%s)" % (ds, r.case, r.side, r.variant))
    for i, (f, (ec, es)) in enumerate(zip(f3d, expected)):
        B, C, D, H, W = f.shape
        ok = (C == ec and H == es and W == es)
        print("  level%d: %s  expect C=%d,HxW=%dx%d  %s"
              % (i, tuple(f.shape), ec, es, es, "OK" if ok else "MISMATCH"))
        assert ok, "feature %d shape mismatch" % i
print("\nAll 3D feature shapes match the decoder contract.")

## 6. Learning curve

In [ ]:
if HISTORY_CSV.exists():
    h = pd.read_csv(HISTORY_CSV)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.epoch, h.loss); ax[0].set_title("SimCLR NT-Xent loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(h.epoch, h.lr); ax[1].set_title("learning rate"); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.show()
else:
    print("no history yet - run the training cell.")

## 7. Visual QA — fused features with the trained encoder

Input DRRs plus a few level-0 fused feature channels for a healthy and a fractured case. With a
fully trained encoder these should show structured, bone-aligned activations (not the near-random
maps of the untrained smoke checkpoint).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for row, (ds, r) in enumerate(samples.items()):
    if r is None:
        continue
    ap = paired_tf(load_drr(r.ap)).unsqueeze(0).to(DEVICE)
    lat = paired_tf(load_drr(r.lat)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        f2d, _ = fusion(ap, lat)
    axes[row, 0].imshow(load_drr(r.ap)[0], cmap="gray"); axes[row, 0].set_title("%s %s\nAP" % (ds, r.case))
    axes[row, 1].imshow(load_drr(r.lat)[0], cmap="gray"); axes[row, 1].set_title("LAT")
    fmap = f2d[0][0].cpu()
    for j in range(3):
        axes[row, 2 + j].imshow(fmap[j], cmap="viridis"); axes[row, 2 + j].set_title("L0 fused ch%d" % j)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout(); plt.show()

## 8. Next steps

1. Copy `models/convnextv2_simclr_encoder.pth` to wherever the decoder runs (it is git-ignored).
2. Run `modeling_HPC/decoder_pipeline.ipynb` — `build_model()` should print
   `loaded SimCLR encoder: missing=0 unexpected=0`. Train `MODEL="unet"`, then `"vnet"`.
3. `FREEZE_ENCODER=False` (decoder default) fine-tunes this encoder jointly with the decoder; set
   `True` to keep it fixed and save memory.
4. To extend the run, set `RESUME_FROM = str(MODELS_DIR / "encoder_simclr_trainstate.pth")` and
   raise `EPOCHS`.